In [1]:
import requests
import json
import dotenv

In [2]:
import os
import requests
from dotenv import load_dotenv

# Load .env file once at app startup (safe to call multiple times)
load_dotenv()


def get_agol_token() -> str:
    """
    Generates an authentication token for ArcGIS Online using a username and password
    stored in environment variables.

    Environment variables required:
        - AGOL_USERNAME
        - AGOL_PASSWORD

    Returns:
        str: A valid authentication token used to make authorized API requests.

    Raises:
        ValueError: If credentials are missing or authentication fails.
        ConnectionError: If requests cannot reach the AGOL endpoint.
    """
    username = os.getenv("AGOL_USERNAME")
    password = os.getenv("AGOL_PASSWORD")

    if not username or not password:
        raise ValueError("Missing AGOL_USERNAME or AGOL_PASSWORD in environment variables.")

    url = "https://www.arcgis.com/sharing/rest/generateToken"

    data = {
        "username": username,
        "password": password,
        "referer": "https://www.arcgis.com",
        "f": "json"
    }

    try:
        response = requests.post(url, data=data)

        if response.status_code != 200:
            raise Exception(
                f"Request failed with status code {response.status_code}: {response.text}"
            )

        token_data = response.json()

        if "token" in token_data:
            return token_data["token"]
        elif "error" in token_data:
            raise ValueError(
                f"Authentication failed: {token_data['error']['message']}"
            )
        else:
            raise ValueError("Unexpected response format: Token not found.")

    except requests.exceptions.RequestException as e:
        raise ConnectionError(f"Failed to connect to ArcGIS Online: {e}")


In [3]:
def query_record(url: str, layer: int, where: str, fields="*", return_geometry=False):
    """
    Executes an SQL-style query against an ArcGIS REST API layer and returns matching records.

    Parameters:
        url: str
            FeatureServer base URL (may or may not already include a layer segment).
        layer: int
            Layer index when url is a FeatureServer root.
        where: str
            SQL-like filter clause (e.g., "GlobalID='...'" or "1=1").
        fields: str
            outFields string. "*" requests all fields.
        return_geometry: bool
            Whether to return geometry in results.

    Returns:
        list: List of 'features' from the ArcGIS REST response.
    """
    token = get_agol_token()
    if not token:
        raise ValueError("Authentication failed: Invalid token.")


    # If the URL already ends with the layer number, don't add it again
    if url.split("/")[-1].isdigit():
        query_url = f"{url}/query"
    else:
        query_url = f"{url}/{layer}/query"

    params = {
        "where": where,
        "outFields": fields,
        "returnGeometry": str(return_geometry).lower(),
        "outSR": 4326,
        "f": "json",
        "token": token
    }

    response = requests.get(query_url, params=params)
    if response.status_code != 200:
        raise Exception(
            f"Request failed with status code {response.status_code}: {response.text}"
        )

    data = response.json()
    if "error" in data:
        raise Exception(
            f"API Error: {data['error']['message']} - {data['error'].get('details', [])}"
        )

    return data.get("features", [])

In [4]:
token = get_agol_token()

traffic_impact_url = "https://services.arcgis.com/r4A0V7UzH9fcLVvv/arcgis/rest/services/service_001ad60c04114d70a690d24123bb6742/FeatureServer"
apex_url = "https://services.arcgis.com/r4A0V7UzH9fcLVvv/arcgis/rest/services/APEX_PROJECTS_LOADER_APPLICATION/FeatureServer"

traffic_impact_layers = {
'traffic_impacts_layer': 0,
'traffic_impact_routes_layer': 1,
'traffic_impact_start_points_layer': 2,
'traffic_impact_end_points_layer': 3
}

apex_layers = {
        "projects_layer": 0,
        "locations_layer": 1,
        "sites_layer": 2,
        "routes_layer": 3,
        "boundaries_layer": 4,
        "impact_comms_layer": 5,
        "region_layer": 6,
        "bor_layer": 7,
        "senate_layer": 8,
        "house_layer": 9,
    }


traffic_impact_form = {
        'traffic_form_url': "https://services.arcgis.com/r4A0V7UzH9fcLVvv/arcgis/rest/services/service_885f75157e3042f2bf3c1cfec1a8094e/FeatureServer",
        'traffic_form_layer': 0
    }

In [5]:
impacts = query_record(
    traffic_impact_form['traffic_form_url'],
    traffic_impact_form['traffic_form_layer'],
    '1=1',
    return_geometry=True
)

In [6]:
success = 0
fail = 0

for impact in impacts:
    objid = impact['attributes']['objectid']
    apex_guid = impact['attributes']['APEX_GUID']
    
    apex_record = query_record(
        apex_url,
        apex_layers['projects_layer'],
        f"globalid='{apex_guid}'",
        return_geometry=True
    )
    
    if len(apex_record) > 0:
        
        region_value = apex_record[0]['attributes']['List_DOT_PF_Region']

        url = f"{traffic_impact_form['traffic_form_url']}/{traffic_impact_form['traffic_form_layer']}/applyEdits"

        payload = {
                    "attributes": {
                        "objectid": objid,
                        "DOT_Region": region_value
                    }
                }

        
        resp = requests.post(
                url,
                data={
                    "f": "json",
                    "token": token,
                    "updates": json.dumps(payload)
                }
            )
        
        result = resp.json()
        update_results = result.get("updateResults", [])
        if update_results and update_results[0].get("success"):
            success += 1
        else:
            fail += 1
        print(update_results)

    else:
        #print(apex_record)
        pass

[{'objectId': 123, 'uniqueId': 123, 'globalId': None, 'success': True}]
[{'objectId': 124, 'uniqueId': 124, 'globalId': None, 'success': True}]
[{'objectId': 125, 'uniqueId': 125, 'globalId': None, 'success': True}]
[{'objectId': 126, 'uniqueId': 126, 'globalId': None, 'success': True}]
[{'objectId': 127, 'uniqueId': 127, 'globalId': None, 'success': True}]
[{'objectId': 128, 'uniqueId': 128, 'globalId': None, 'success': True}]
[{'objectId': 129, 'uniqueId': 129, 'globalId': None, 'success': True}]
[{'objectId': 130, 'uniqueId': 130, 'globalId': None, 'success': True}]
[{'objectId': 131, 'uniqueId': 131, 'globalId': None, 'success': True}]
[{'objectId': 132, 'uniqueId': 132, 'globalId': None, 'success': True}]
[{'objectId': 133, 'uniqueId': 133, 'globalId': None, 'success': True}]
[{'objectId': 134, 'uniqueId': 134, 'globalId': None, 'success': True}]
[{'objectId': 135, 'uniqueId': 135, 'globalId': None, 'success': True}]
[{'objectId': 136, 'uniqueId': 136, 'globalId': None, 'success':